In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""

In [2]:
import pandas as pd
import re
import csv
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

In [3]:
df = pd.read_csv("dataset.csv")


df, _ = train_test_split(
    df,
    train_size=10000,
    stratify=df["label"],
    random_state=42
)

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

print("Treino:", len(train_df))
print("Validação:", len(val_df))
print("Teste:", len(test_df))

Treino: 7000
Validação: 1500
Teste: 1500


In [4]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

/home/gaby/.local/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")
precision = evaluate.load("precision")
recall = evaluate.load("recall")

In [6]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy.compute(
            predictions=predictions,
            references=labels
        )["accuracy"],

        "f1": f1.compute(
            predictions=predictions,
            references=labels,
            average="binary"
        )["f1"],

        "precision": precision.compute(
            predictions=predictions,
            references=labels,
            average="binary"
        )["precision"],

        "recall": recall.compute(
            predictions=predictions,
            references=labels,
            average="binary"
        )["recall"]
    }

In [7]:
models = [
    "distilbert-base-uncased",
    "bert-base-uncased",
    "roberta-base"
]

In [34]:
results = []

for model_name in models:

    print(f"\nTreinando {model_name}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            padding="max_length",
            max_length=64
        )

    train_tok = train_dataset.map(tokenize, batched=True)
    val_tok = val_dataset.map(tokenize, batched=True)
    test_tok = test_dataset.map(tokenize, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2
    )

    args = TrainingArguments(
    output_dir=f"./{model_name.split('/')[-1]}",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    gradient_accumulation_steps=8,

    num_train_epochs=3,

    load_best_model_at_end=True,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        compute_metrics=compute_metrics
    )

    trainer.train()

    metrics = trainer.evaluate(test_tok)

    results.append({
        "modelo": model_name,
        "accuracy": metrics["eval_accuracy"],
        "f1": metrics["eval_f1"],
        "precision": metrics["eval_precision"],
        "recall": metrics["eval_recall"]
    })


Treinando distilbert-base-uncased


Map: 100%|██████████| 67345/67345 [00:16<00:00, 4103.06 examples/s]
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
  1%|          | 500/58926 [04:56<9:56:26,  1.63it/s] 

{'loss': 0.0753, 'grad_norm': 0.15075770020484924, 'learning_rate': 1.9830295625021216e-05, 'epoch': 0.03}


  2%|▏         | 1000/58926 [09:54<9:27:55,  1.70it/s]

{'loss': 0.0613, 'grad_norm': 0.031512778252363205, 'learning_rate': 1.9660591250042427e-05, 'epoch': 0.05}


...

{'train_runtime': 38054.7215, 'train_samples_per_second': 24.775, 'train_steps_per_second': 1.548, 'train_loss': 0.02357871116989965, 'epoch': 3.0}


100%|██████████| 33673/33673 [12:09<00:00, 46.14it/s]



Treinando bert-base-uncased


Map: 100%|██████████| 67345/67345 [00:17<00:00, 3804.55 examples/s]
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
  0%|          | 0/58926 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 1.95 GiB of which 4.50 MiB is free. Including non-PyTorch memory, this process has 1.58 GiB memory in use. Of the allocated memory 1.40 GiB is allocated by PyTorch, and 131.62 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [8]:
import gc
import torch

results = []


print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Treinando em CPU")

for model_name in models:

    print(f"\nTreinando {model_name}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            padding="max_length",

            # ALTERAÇÃO: reduzir tamanho máximo dos textos
            # 256 -> 64 para diminuir drasticamente o consumo de VRAM
            max_length=64
        )

    train_tok = train_dataset.map(tokenize, batched=True)
    val_tok = val_dataset.map(tokenize, batched=True)
    test_tok = test_dataset.map(tokenize, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2
    )
    
    model.to("cpu")

    # ALTERAÇÃO: ativa Gradient Checkpointing
    # reduz uso de memória durante o treinamento
    model.gradient_checkpointing_enable()

    args = TrainingArguments(

        # ALTERAÇÃO:
        # evita criar pasta com o mesmo nome do modelo
        # que pode causar conflito posteriormente
        output_dir=f"./checkpoints/{model_name.split('/')[-1]}",

        eval_strategy="epoch",
        save_strategy="epoch",

        learning_rate=2e-5,

        # ALTERAÇÃO:
        # batch reduzido para economizar memória
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,

        # ALTERAÇÃO:
        # compensa o batch pequeno
        gradient_accumulation_steps=4,

        # ALTERAÇÃO:
        # utiliza precisão mista (se a GPU suportar)
        fp16=False,

        num_train_epochs=3,

        load_best_model_at_end=True,

        # ALTERAÇÃO:
        # evita logs desnecessários
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        compute_metrics=compute_metrics
    )

    trainer.train()

    metrics = trainer.evaluate(test_tok)

    results.append({
        "modelo": model_name,
        "accuracy": metrics["eval_accuracy"],
        "f1": metrics["eval_f1"],
        "precision": metrics["eval_precision"],
        "recall": metrics["eval_recall"]
    })

    # ALTERAÇÃO:
    # libera memória da GPU após cada treinamento
    del trainer
    del model

    gc.collect()
    torch.cuda.empty_cache()

CUDA disponível: False
Treinando em CPU

Treinando distilbert-base-uncased


Map: 100%|██████████| 1500/1500 [00:00<00:00, 4607.62 examples/s]
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
  0%|          | 0/1311 [00:00<?, ?it/s]/home/gaby/.local/lib/python3.8/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]
 33%|███▎      | 437/1311 [21:01<43:23,  2.98s/it]  /home/gaby/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1272: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due 

{'eval_loss': 0.06140151992440224, 'eval_accuracy': 0.988, 'eval_f1': 0.0, 'eval_precision': 0.0, 'eval_recall': 0.0, 'eval_runtime': 63.4718, 'eval_samples_per_second': 23.633, 'eval_steps_per_second': 5.908, 'epoch': 1.0}


/home/gaby/.local/lib/python3.8/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]
 38%|███▊      | 500/1311 [25:09<40:19,  2.98s/it]  

{'loss': 0.0718, 'grad_norm': 0.052331361919641495, 'learning_rate': 1.2372234935163998e-05, 'epoch': 1.14}


...

100%|██████████| 1311/1311 [2:08:04<00:00,  5.86s/it]


{'train_runtime': 7684.475, 'train_samples_per_second': 2.733, 'train_steps_per_second': 0.171, 'train_loss': 0.05817586970092867, 'epoch': 3.0}


100%|██████████| 375/375 [01:57<00:00,  3.19it/s]



Treinando microsoft/deberta-v3-base


ValueError: Converting from Tiktoken failed, if a converter for SentencePiece is available, provide a model path with a SentencePiece tokenizer.model file.Currently available slow->fast convertors: ['AlbertTokenizer', 'BartTokenizer', 'BarthezTokenizer', 'BertTokenizer', 'BigBirdTokenizer', 'BlenderbotTokenizer', 'CamembertTokenizer', 'CLIPTokenizer', 'CodeGenTokenizer', 'ConvBertTokenizer', 'DebertaTokenizer', 'DebertaV2Tokenizer', 'DistilBertTokenizer', 'DPRReaderTokenizer', 'DPRQuestionEncoderTokenizer', 'DPRContextEncoderTokenizer', 'ElectraTokenizer', 'FNetTokenizer', 'FunnelTokenizer', 'GPT2Tokenizer', 'HerbertTokenizer', 'LayoutLMTokenizer', 'LayoutLMv2Tokenizer', 'LayoutLMv3Tokenizer', 'LayoutXLMTokenizer', 'LongformerTokenizer', 'LEDTokenizer', 'LxmertTokenizer', 'MarkupLMTokenizer', 'MBartTokenizer', 'MBart50Tokenizer', 'MPNetTokenizer', 'MobileBertTokenizer', 'MvpTokenizer', 'NllbTokenizer', 'OpenAIGPTTokenizer', 'PegasusTokenizer', 'Qwen2Tokenizer', 'RealmTokenizer', 'Refor

In [9]:

results_df = pd.DataFrame(results)

print(
    results_df.sort_values(
        by="f1",
        ascending=False
    )
)

                    modelo  accuracy        f1  precision    recall
1        bert-base-uncased  0.990667  0.533333   0.666667  0.444444
0  distilbert-base-uncased  0.988000  0.400000   0.500000  0.333333
2             roberta-base  0.990000  0.347826   0.800000  0.222222


In [19]:
print(model.config.vocab_size)

30522
